# Prepare Gold Labels

This notebook keeps the gold-label workflow intentionally simple.

It does only three things:

1. Load the base feature dataset.
2. Define a single hardcoded dictionary of gold labels grouped by department.
3. Flatten, standardize, save, and join those labels back onto the feature dataset.

There is no candidate sampling, no statistical selection logic, and no debug augmentation in this version.


## 1. Environment Setup

The notebook loads the project paths and reads the base feature dataset that the gold labels will be joined onto.


In [ ]:
from pathlib import Path
from datetime import date

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "Data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", DATA_PROCESSED)


In [ ]:
FEATURE_DATA_PATH = DATA_PROCESSED / "contracts_with_features.csv"

if not FEATURE_DATA_PATH.exists():
    FEATURE_DATA_PATH = DATA_PROCESSED / "contract_with_features_labeled.csv"

if not FEATURE_DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find either contracts_with_features.csv or contract_with_features_labeled.csv "
        "in Data/processed."
    )

df_features = pd.read_csv(FEATURE_DATA_PATH, low_memory=False)

print("Loaded feature dataset from:", FEATURE_DATA_PATH)
print("Shape:", df_features.shape)
display(df_features.head())


## 2. Hardcoded Gold Label Dictionary

Add all gold labels here, grouped by department.

Rules:
- Keep `contract_id` and `contract_number` as strings.
- Use `gold_y = 1` for positive and `gold_y = 0` for negative.
- Keep the department names exactly as they appear in the feature dataset when possible.
- Add or remove departments freely. Empty department lists are allowed.


In [ ]:
gold_label_records = [
    # Bioprocessing
    {"contract_id": "504361", "contract_number": "504149", "gold_y": 1, "department": "Bioprocessing and Excipients"},
    {"contract_id": "613927", "contract_number": "612555", "gold_y": 1, "department": "Bioprocessing and Excipients"},
    {"contract_id": "345318", "contract_number": "346312", "gold_y": 0, "department": "Bioprocessing and Excipients"},
    {"contract_id": "1877",   "contract_number": "1877",   "gold_y": 0, "department": "Bioprocessing and Excipients"},
    
    # Global Strategic
    {"contract_id": "4821",   "contract_number": "4822",   "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "557",    "contract_number": "557",    "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "565",    "contract_number": "565",    "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "483149", "contract_number": "483178", "gold_y": 0, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "25777",  "contract_number": "25777",  "gold_y": 0, "department": "Global Strategic Outsourcing & Devices & Needles"},
    
    # Aseptic
    {"contract_id": "577893", "contract_number": "577893", "gold_y": 0, "department": "Aseptic Production"},
    {"contract_id": "370849", "contract_number": "370849", "gold_y": 0, "department": "Aseptic Production"},
    {"contract_id": "595207", "contract_number": "595207", "gold_y": 0, "department": "Aseptic Production"},
    
    # Packaging
    {"contract_id": "193",  "contract_number": "193",  "gold_y": 1, "department": "Packaging material"}, 
    {"contract_id": "213",  "contract_number": "213",  "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "190",  "contract_number": "190",  "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1265", "contract_number": "1265", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1245", "contract_number": "1245", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1256", "contract_number": "1256", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "248",  "contract_number": "248",  "gold_y": 0, "department": "Packaging material"},
    {"contract_id": "250",  "contract_number": "250",  "gold_y": 0, "department": "Packaging material"},
    
    # Logistics
    {"contract_id": "192312", "contract_number": "192325", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "541883", "contract_number": "541269", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "582913", "contract_number": "581732", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "569028", "contract_number": "568080", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "470427", "contract_number": "470608", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "530290", "contract_number": "529779", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "533151", "contract_number": "532634", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "550918", "contract_number": "550237", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "525275", "contract_number": "524758", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "335471", "contract_number": "336520", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "607710", "contract_number": "606340", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "536891", "contract_number": "536319", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "599608", "contract_number": "598142", "gold_y": 0, "department": "Logistics"},
]

In [ ]:
gold_labels_by_department = {
    "Logistics": [
        # Example:
        # {
        #     "contract_id": "12345",
        #     "contract_number": "CN-001",
        #     "gold_y": 1,
        #     "notes": "Manual review comment"
        # }
    ],
    "Packaging Material": [],
    "Devices & Needles": [],
    "Drug Product Outsourcing": [],
    "Drug Substance Outsourcing": [],
    "Raw Materials & Energy": [],
    "Quality, Production Services & Supplies": [],
    "Bioprocessing and Excipients": [],
    "Bioprocessing & Raw Materials": [],
    "Alliance, Acquisitions & PPM CoE": [],
    "Global Strategic Outsourcing & Devices": [],
    "Strategic Sourcing US Hub": [],
    "Strategy, Sourcing & Negotiation CoE": [],
    "HI Warehouse Expansion Program": [],
    "Aseptic Production": [],
}

def flatten_gold_labels(gold_dict: dict) -> list[dict]:
    records = []
    today = str(date.today())

    for department, rows in gold_dict.items():
        for row in rows:
            record = row.copy()
            record["department"] = department
            record["label_source"] = record.get("label_source", "manual_hardcoded")
            record["label_date"] = record.get("label_date", today)
            record["notes"] = record.get("notes", "")
            records.append(record)

    return records

gold_label_records = flatten_gold_labels(gold_labels_by_department)

df_gold = pd.DataFrame(gold_label_records)

if df_gold.empty:
    df_gold = pd.DataFrame(
        columns=[
            "contract_id",
            "contract_number",
            "department",
            "gold_y",
            "label_source",
            "label_date",
            "notes",
        ]
    )

print("Hardcoded gold label rows:", len(df_gold))
display(df_gold.head())


## 3. Standardize and Save the Gold Labels

This section standardizes the schema, removes exact duplicates, and saves the clean gold-label table.


In [ ]:
required_cols = [
    "contract_id",
    "contract_number",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]

for col in required_cols:
    if col not in df_gold.columns:
        df_gold[col] = pd.NA

df_gold = df_gold[required_cols].copy()

for col in ["contract_id", "contract_number", "department"]:
    df_gold[col] = (
        df_gold[col]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

df_gold["gold_y"] = pd.to_numeric(df_gold["gold_y"], errors="coerce")

invalid_labels = df_gold[~df_gold["gold_y"].isin([0, 1]) & df_gold["gold_y"].notna()]
if not invalid_labels.empty:
    raise ValueError("Invalid gold_y values found. Only 0 and 1 are allowed.")

df_gold["gold_y"] = df_gold["gold_y"].astype("Int64")
df_gold["label_source"] = df_gold["label_source"].fillna("manual_hardcoded")
df_gold["label_date"] = df_gold["label_date"].fillna(str(date.today()))
df_gold["notes"] = df_gold["notes"].fillna("")

df_gold_clean = df_gold.drop_duplicates().copy()

GOLD_CLEAN_PATH = DATA_PROCESSED / "gold_labels_clean.csv"
df_gold_clean.to_csv(GOLD_CLEAN_PATH, index=False)

print("Saved clean gold labels to:", GOLD_CLEAN_PATH)
print("Shape:", df_gold_clean.shape)
display(df_gold_clean.head())


## 4. Join Gold Labels onto the Feature Dataset

This join creates the modeling-ready dataset with `gold_y` attached to the feature table.


In [ ]:
df_features_joined = df_features.copy()

for col in ["contract_id", "contract_number"]:
    if col in df_features_joined.columns:
        df_features_joined[col] = (
            df_features_joined[col]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

for col in ["gold_y", "gold_department", "label_source", "label_date", "notes"]:
    if col in df_features_joined.columns:
        df_features_joined = df_features_joined.drop(columns=[col])

join_cols = [
    "contract_id",
    "contract_number",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]

df_gold_for_join = df_gold_clean[join_cols].copy()

df_features_with_gold = df_features_joined.merge(
    df_gold_for_join.drop(columns=["contract_number"], errors="ignore"),
    on="contract_id",
    how="left",
    suffixes=("", "_gold"),
)

if "department_gold" in df_features_with_gold.columns:
    df_features_with_gold = df_features_with_gold.rename(columns={"department_gold": "gold_department"})
else:
    df_features_with_gold["gold_department"] = pd.NA

FEATURES_WITH_GOLD_PATH = DATA_PROCESSED / "contract_with_features_gold_joined.csv"
df_features_with_gold.to_csv(FEATURES_WITH_GOLD_PATH, index=False)

print("Saved feature dataset with gold labels to:", FEATURES_WITH_GOLD_PATH)
print("Shape:", df_features_with_gold.shape)
print("Gold-labeled rows:", int(df_features_with_gold["gold_y"].notna().sum()))
display(df_features_with_gold.head())


## 5. Quick Summary

This final cell gives a compact overview of how many gold labels have been hardcoded and which departments currently contain labels.


In [ ]:
if df_gold_clean.empty:
    print("No hardcoded gold labels have been added yet.")
else:
    summary = (
        df_gold_clean.groupby("department", dropna=False)
        .agg(
            gold_total=("gold_y", "count"),
            gold_yes=("gold_y", "sum"),
        )
        .reset_index()
    )
    summary["gold_no"] = summary["gold_total"] - summary["gold_yes"]
    summary = summary.sort_values(["gold_total", "department"], ascending=[False, True]).reset_index(drop=True)

    display(summary)

    print("Total gold labels:", len(df_gold_clean))
    print("Departments with labels:", summary["department"].nunique())
